# 02 — Twitch Ranking Data Preparation & Feature Engineering

This notebook transforms the raw Twitch interaction log into **leakage-safe training and test tables** for a personalized recommendation/ranking model.

## Prediction task

For a user \(u\) and candidate streamer \(s\), estimate:

\[
P(	ext{user watches streamer in the future} \mid 	ext{historical behavior})
\]

The model will eventually generate an inference score for each candidate streamer, and the ranking layer will sort those scores to produce Top-K recommendations.

## What this notebook does

1. Load and clean Twitch interactions
2. Create chronological observation/target windows
3. Engineer user behavior features
4. Engineer streamer behavior features
5. Engineer user-streamer affinity features
6. Engineer recency and repeat-consumption features
7. Engineer short-term popularity/trend features
8. Build positive future targets
9. Generate negative candidates
10. Join candidates with historical features
11. Run leakage/sanity checks
12. Save processed train/test ranking tables

> Twitch contains implicit-feedback logs, not impression logs. Therefore, negative samples are **unobserved candidate streamers**, not confirmed dislikes.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


## 1. Load the raw Twitch data

In [2]:
DATA_PATH = "../data/raw/twitch/100k_a.csv"

columns = [
    "user_id",
    "stream_id",
    "streamer",
    "start_time",
    "stop_time",
]

df = pd.read_csv(
    DATA_PATH,
    names=columns,
)

print(f"Rows: {len(df):,}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Streamers: {df['streamer'].nunique():,}")
print(f"Streams: {df['stream_id'].nunique():,}")

df.head()


Rows: 3,051,733
Users: 100,000
Streamers: 162,625
Streams: 739,991


,user_id,stream_id,streamer,start_time,stop_time
0,1,33842865744,mithrain,154,156
1,1,33846768288,alptv,166,169
2,1,33886469056,mithrain,587,588
3,1,33887624992,wtcn,589,591
4,1,33890145056,jrokezftw,591,594


## 2. Clean data and derive watch duration

`start_time` and `stop_time` are expressed in 10-minute intervals.


In [3]:
df["duration_intervals"] = df["stop_time"] - df["start_time"]
df["watch_minutes"] = df["duration_intervals"] * 10

before = len(df)

df = df[
    df["user_id"].notna()
    & df["streamer"].notna()
    & df["start_time"].notna()
    & df["stop_time"].notna()
    & (df["duration_intervals"] > 0)
].copy()

after = len(df)

print(f"Removed invalid rows: {before - after:,}")
print(f"Remaining rows: {after:,}")
print(f"Median watch minutes: {df['watch_minutes'].median():,.0f}")

df.head()


Removed invalid rows: 0
Remaining rows: 3,051,733
Median watch minutes: 10


,user_id,stream_id,streamer,start_time,stop_time,duration_intervals,watch_minutes
0,1,33842865744,mithrain,154,156,2,20
1,1,33846768288,alptv,166,169,3,30
2,1,33886469056,mithrain,587,588,1,10
3,1,33887624992,wtcn,589,591,2,20
4,1,33890145056,jrokezftw,591,594,3,30


## 3. Sort chronologically

In [4]:
df = df.sort_values(
    ["start_time", "user_id", "streamer"]
).reset_index(drop=True)

print("Earliest start interval:", df["start_time"].min())
print("Latest start interval:  ", df["start_time"].max())


Earliest start interval: 0
Latest start interval:   6147


## 4. Create chronological windows

We use:

- **0–70% of time** → training history/features
- **70–85% of time** → training target window
- **0–85% of time** → test history/features
- **85–100% of time** → final test target window

This avoids random-split leakage.


In [5]:
cutoff_train = int(np.floor(df["start_time"].quantile(0.70)))
cutoff_test = int(np.floor(df["start_time"].quantile(0.85)))

history_train = df[df["start_time"] < cutoff_train].copy()

target_train = df[
    (df["start_time"] >= cutoff_train)
    & (df["start_time"] < cutoff_test)
].copy()

history_test = df[df["start_time"] < cutoff_test].copy()

target_test = df[
    df["start_time"] >= cutoff_test
].copy()

print("Train feature cutoff:", cutoff_train)
print("Test feature cutoff: ", cutoff_test)
print()
print(f"Training history interactions: {len(history_train):,}")
print(f"Training target interactions:  {len(target_train):,}")
print(f"Test history interactions:     {len(history_test):,}")
print(f"Test target interactions:      {len(target_test):,}")


Train feature cutoff: 4376
Test feature cutoff:  5288

Training history interactions: 2,136,161
Training target interactions:  457,374
Test history interactions:     2,593,535
Test target interactions:      458,198


## 5. Warm-user coverage

For the first model we focus on **warm users**: users who have historical behavior before the cutoff and also appear in the future prediction window.

Cold-start users will be handled separately later.


In [6]:
train_history_users = set(history_train["user_id"].unique())
train_target_users = set(target_train["user_id"].unique())

test_history_users = set(history_test["user_id"].unique())
test_target_users = set(target_test["user_id"].unique())

eligible_train_users = train_history_users & train_target_users
eligible_test_users = test_history_users & test_target_users

cold_train_users = train_target_users - train_history_users
cold_test_users = test_target_users - test_history_users

print(f"Eligible train users: {len(eligible_train_users):,}")
print(f"Cold-start users in train target: {len(cold_train_users):,}")
print()
print(f"Eligible test users: {len(eligible_test_users):,}")
print(f"Cold-start users in test target: {len(cold_test_users):,}")


Eligible train users: 71,294
Cold-start users in train target: 2,219

Eligible test users: 71,164
Cold-start users in test target: 1,198


## 6. Development-size control

Use a subset while validating the pipeline. Later set both values to `None` to build the full data.


In [7]:
MAX_USERS_TRAIN = 20_000
MAX_USERS_TEST = 10_000

def sample_users(user_set, max_users, seed):
    users = np.array(sorted(user_set))

    if max_users is None or len(users) <= max_users:
        return users

    local_rng = np.random.default_rng(seed)

    return local_rng.choice(
        users,
        size=max_users,
        replace=False,
    )

train_users = sample_users(
    eligible_train_users,
    MAX_USERS_TRAIN,
    RANDOM_SEED,
)

test_users = sample_users(
    eligible_test_users,
    MAX_USERS_TEST,
    RANDOM_SEED + 1,
)

print(f"Selected train users: {len(train_users):,}")
print(f"Selected test users:  {len(test_users):,}")


Selected train users: 20,000
Selected test users:  10,000


# 7. Feature Engineering

All model features are computed **strictly from the history available before the prediction cutoff**.

We will create four feature groups:

1. User behavior features
2. Streamer behavior/popularity features
3. User-streamer affinity features
4. Temporal/recency/trend features


## 7.1 User behavior features

In [8]:
def build_user_features(history: pd.DataFrame, cutoff: int):
    base = (
        history.groupby("user_id")
        .agg(
            user_interactions=("streamer", "size"),
            user_unique_streamers=("streamer", "nunique"),
            user_total_watch_minutes=("watch_minutes", "sum"),
            user_avg_watch_minutes=("watch_minutes", "mean"),
            user_median_watch_minutes=("watch_minutes", "median"),
            user_first_start=("start_time", "min"),
            user_last_start=("start_time", "max"),
        )
        .reset_index()
    )

    base["user_active_span_intervals"] = (
        base["user_last_start"] - base["user_first_start"]
    )

    base["user_last_activity_recency"] = (
        cutoff - base["user_last_start"]
    ).clip(lower=0)

    base["user_repeat_rate"] = (
        1
        - (
            base["user_unique_streamers"]
            / base["user_interactions"].clip(lower=1)
        )
    )

    # Recent user activity
    for window in [6, 18, 42]:
        recent = history[
            history["start_time"] >= cutoff - window
        ]

        recent_counts = (
            recent.groupby("user_id")
            .size()
            .rename(f"user_interactions_last_{window}_intervals")
            .reset_index()
        )

        base = base.merge(
            recent_counts,
            on="user_id",
            how="left",
        )

        base[f"user_interactions_last_{window}_intervals"] = (
            base[f"user_interactions_last_{window}_intervals"]
            .fillna(0)
        )

    return base


train_user_features = build_user_features(
    history_train,
    cutoff_train,
)

test_user_features = build_user_features(
    history_test,
    cutoff_test,
)

train_user_features.head()


,user_id,user_interactions,user_unique_streamers,user_total_watch_minutes,user_avg_watch_minutes,user_median_watch_minutes,user_first_start,user_last_start,user_active_span_intervals,user_last_activity_recency,user_repeat_rate,user_interactions_last_6_intervals,user_interactions_last_18_intervals,user_interactions_last_42_intervals
0,1,27,13,590,21.8519,10.0000,154,4327,4173,49,0.5185,0.0000,0.0000,0.0000
1,2,28,12,470,16.7857,10.0000,58,4280,4222,96,0.5714,0.0000,0.0000,0.0000
2,3,9,5,140,15.5556,10.0000,135,2578,2443,1798,0.4444,0.0000,0.0000,0.0000
3,4,57,20,920,16.1404,10.0000,142,4136,3994,240,0.6491,0.0000,0.0000,0.0000
4,5,4,1,140,35.0000,30.0000,16,3595,3579,781,0.7500,0.0000,0.0000,0.0000


The 6/18/42-interval windows represent approximately 1, 3, and 7 hours because one interval is 10 minutes. These are short-horizon activity features rather than literal calendar-day windows.


## 7.2 Streamer popularity and trend features

In [9]:
def build_streamer_features(history: pd.DataFrame, cutoff: int):
    base = (
        history.groupby("streamer")
        .agg(
            streamer_interactions=("user_id", "size"),
            streamer_unique_viewers=("user_id", "nunique"),
            streamer_total_watch_minutes=("watch_minutes", "sum"),
            streamer_avg_watch_minutes=("watch_minutes", "mean"),
        )
        .reset_index()
    )

    total_interactions = max(len(history), 1)

    base["streamer_popularity_share"] = (
        base["streamer_interactions"]
        / total_interactions
    )

    base["streamer_log_interactions"] = np.log1p(
        base["streamer_interactions"]
    )

    base["streamer_log_viewers"] = np.log1p(
        base["streamer_unique_viewers"]
    )

    # Repeat-viewer rate:
    # fraction of streamer viewers who interacted with streamer >1 time
    pair_counts = (
        history.groupby(["streamer", "user_id"])
        .size()
        .reset_index(name="pair_count")
    )

    repeat_viewer_rate = (
        pair_counts.assign(
            repeated=(pair_counts["pair_count"] > 1).astype(int)
        )
        .groupby("streamer")["repeated"]
        .mean()
        .rename("streamer_repeat_viewer_rate")
        .reset_index()
    )

    base = base.merge(
        repeat_viewer_rate,
        on="streamer",
        how="left",
    )

    # Short-term popularity
    recent_6 = (
        history[history["start_time"] >= cutoff - 6]
        .groupby("streamer")
        .size()
        .rename("streamer_interactions_last_6")
        .reset_index()
    )

    recent_18 = (
        history[history["start_time"] >= cutoff - 18]
        .groupby("streamer")
        .size()
        .rename("streamer_interactions_last_18")
        .reset_index()
    )

    recent_42 = (
        history[history["start_time"] >= cutoff - 42]
        .groupby("streamer")
        .size()
        .rename("streamer_interactions_last_42")
        .reset_index()
    )

    previous_42 = (
        history[
            (history["start_time"] >= cutoff - 84)
            & (history["start_time"] < cutoff - 42)
        ]
        .groupby("streamer")
        .size()
        .rename("streamer_interactions_prev_42")
        .reset_index()
    )

    for recent_df in [
        recent_6,
        recent_18,
        recent_42,
        previous_42,
    ]:
        base = base.merge(
            recent_df,
            on="streamer",
            how="left",
        )

    count_cols = [
        "streamer_interactions_last_6",
        "streamer_interactions_last_18",
        "streamer_interactions_last_42",
        "streamer_interactions_prev_42",
    ]

    base[count_cols] = base[count_cols].fillna(0)

    # Trend ratio:
    # >1 = more active recently, <1 = losing activity
    base["streamer_trend_ratio"] = (
        (base["streamer_interactions_last_42"] + 1)
        / (base["streamer_interactions_prev_42"] + 1)
    )

    return base


train_streamer_features = build_streamer_features(
    history_train,
    cutoff_train,
)

test_streamer_features = build_streamer_features(
    history_test,
    cutoff_test,
)

train_streamer_features.head()


,streamer,streamer_interactions,streamer_unique_viewers,streamer_total_watch_minutes,streamer_avg_watch_minutes,streamer_popularity_share,streamer_log_interactions,streamer_log_viewers,streamer_repeat_viewer_rate,streamer_interactions_last_6,streamer_interactions_last_18,streamer_interactions_last_42,streamer_interactions_prev_42,streamer_trend_ratio
0,000aikia000,1,1,10,10.0000,0.0000,0.6931,0.6931,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,0011am,1,1,80,80.0000,0.0000,0.6931,0.6931,0.0000,0.0000,0.0000,0.0000,1.0000,0.5000
2,001_ssmashh_100,3,1,50,16.6667,0.0000,1.3863,0.6931,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000
3,002ivorkdorog,1,1,40,40.0000,0.0000,0.6931,0.6931,0.0000,0.0000,0.0000,0.0000,1.0000,0.5000
4,007kingchannel,1,1,10,10.0000,0.0000,0.6931,0.6931,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000


## 7.3 User-streamer affinity features

In [10]:
def build_pair_features(history: pd.DataFrame, cutoff: int):
    pair = (
        history.groupby(["user_id", "streamer"])
        .agg(
            pair_interactions=("stream_id", "size"),
            pair_watch_minutes=("watch_minutes", "sum"),
            pair_avg_watch_minutes=("watch_minutes", "mean"),
            pair_last_start=("start_time", "max"),
            pair_first_start=("start_time", "min"),
        )
        .reset_index()
    )

    pair["pair_recency_intervals"] = (
        cutoff - pair["pair_last_start"]
    ).clip(lower=0)

    pair["pair_active_span_intervals"] = (
        pair["pair_last_start"] - pair["pair_first_start"]
    )

    pair["pair_repeat_strength"] = np.log1p(
        pair["pair_interactions"]
    )

    # recent pair interactions
    for window in [6, 18, 42]:
        recent = history[
            history["start_time"] >= cutoff - window
        ]

        recent_counts = (
            recent.groupby(["user_id", "streamer"])
            .size()
            .rename(f"pair_interactions_last_{window}")
            .reset_index()
        )

        pair = pair.merge(
            recent_counts,
            on=["user_id", "streamer"],
            how="left",
        )

        pair[f"pair_interactions_last_{window}"] = (
            pair[f"pair_interactions_last_{window}"]
            .fillna(0)
        )

    return pair


train_pair_features = build_pair_features(
    history_train,
    cutoff_train,
)

test_pair_features = build_pair_features(
    history_test,
    cutoff_test,
)

train_pair_features.head()


,user_id,streamer,pair_interactions,pair_watch_minutes,pair_avg_watch_minutes,pair_last_start,pair_first_start,pair_recency_intervals,pair_active_span_intervals,pair_repeat_strength,pair_interactions_last_6,pair_interactions_last_18,pair_interactions_last_42
0,1,alptv,1,30,30.0000,166,166,4210,0,0.6931,0.0000,0.0000,0.0000
1,1,berkriptepe,1,30,30.0000,734,734,3642,0,0.6931,0.0000,0.0000,0.0000
2,1,elraenn,2,20,10.0000,4314,2600,62,1714,1.0986,0.0000,0.0000,0.0000
3,1,eraymaskulen,1,10,10.0000,4327,4327,49,0,0.6931,0.0000,0.0000,0.0000
4,1,jahrein,1,30,30.0000,3757,3757,619,0,0.6931,0.0000,0.0000,0.0000


## 7.4 Feature-table sizes


In [11]:
print("TRAIN")
print("User features:    ", train_user_features.shape)
print("Streamer features:", train_streamer_features.shape)
print("Pair features:    ", train_pair_features.shape)

print()
print("TEST")
print("User features:    ", test_user_features.shape)
print("Streamer features:", test_streamer_features.shape)
print("Pair features:    ", test_pair_features.shape)


TRAIN
User features:     (96583, 14)
Streamer features: (131994, 14)
Pair features:     (1155533, 13)

TEST
User features:     (98802, 14)
Streamer features: (148585, 14)
Pair features:     (1337200, 13)


## 8. Build future positive targets

A positive pair means the user actually watched that streamer during the future target window.


In [12]:
def build_positive_pairs(target: pd.DataFrame, users):
    selected = target[
        target["user_id"].isin(users)
    ].copy()

    positives = (
        selected.groupby(["user_id", "streamer"])
        .agg(
            future_interactions=("stream_id", "size"),
            future_watch_minutes=("watch_minutes", "sum"),
        )
        .reset_index()
    )

    positives["label"] = 1

    return positives


train_positives = build_positive_pairs(
    target_train,
    train_users,
)

test_positives = build_positive_pairs(
    target_test,
    test_users,
)

print(f"Training positive pairs: {len(train_positives):,}")
print(f"Test positive pairs:     {len(test_positives):,}")

train_positives.head()


Training positive pairs: 99,228
Test positive pairs:     48,121


,user_id,streamer,future_interactions,future_watch_minutes,label
0,1,esl_csgo,1,10,1
1,1,h3x_tv,1,10,1
2,1,jahrein,1,20,1
3,1,jrokezftw,1,10,1
4,1,ogrencievi,2,40,1


## 9. Candidate generation and negative sampling

Because Twitch has no impression log, we do not know which streamers were shown and ignored.

We therefore sample candidate negatives from streamers that were sufficiently active before the cutoff.

This creates a realistic supervised ranking table while preserving the distinction between:

- positive = future observed watch
- negative = sampled unobserved candidate


In [13]:
MIN_STREAMER_INTERACTIONS = 5
NEGATIVES_PER_POSITIVE = 4

def build_candidate_pool(streamer_features):
    return streamer_features.loc[
        streamer_features["streamer_interactions"]
        >= MIN_STREAMER_INTERACTIONS,
        "streamer",
    ].to_numpy()


train_candidate_pool = build_candidate_pool(
    train_streamer_features
)

test_candidate_pool = build_candidate_pool(
    test_streamer_features
)

print(f"Train candidate pool: {len(train_candidate_pool):,}")
print(f"Test candidate pool:  {len(test_candidate_pool):,}")


Train candidate pool: 33,788
Test candidate pool:  38,920


In [14]:
def build_candidate_labels(
    positives: pd.DataFrame,
    candidate_pool: np.ndarray,
    negatives_per_positive: int,
    seed: int,
):
    local_rng = np.random.default_rng(seed)

    positive_lookup = (
        positives.groupby("user_id")["streamer"]
        .apply(set)
        .to_dict()
    )

    positive_rows = positives[
        ["user_id", "streamer", "label"]
    ].copy()

    negative_rows = []

    for user_id, positive_streamers in positive_lookup.items():

        n_positive = len(positive_streamers)

        n_negative = max(
            negatives_per_positive * n_positive,
            negatives_per_positive,
        )

        available = np.array([
            s
            for s in candidate_pool
            if s not in positive_streamers
        ])

        if len(available) == 0:
            continue

        n_negative = min(
            n_negative,
            len(available),
        )

        sampled = local_rng.choice(
            available,
            size=n_negative,
            replace=False,
        )

        for streamer in sampled:
            negative_rows.append(
                {
                    "user_id": user_id,
                    "streamer": streamer,
                    "label": 0,
                }
            )

    negatives = pd.DataFrame(negative_rows)

    candidates = pd.concat(
        [positive_rows, negatives],
        ignore_index=True,
    )

    candidates = candidates.drop_duplicates(
        ["user_id", "streamer"]
    )

    return candidates


train_candidates = build_candidate_labels(
    train_positives,
    train_candidate_pool,
    NEGATIVES_PER_POSITIVE,
    RANDOM_SEED,
)

test_candidates = build_candidate_labels(
    test_positives,
    test_candidate_pool,
    NEGATIVES_PER_POSITIVE,
    RANDOM_SEED + 1,
)

print(f"Train candidate rows: {len(train_candidates):,}")
print(f"Test candidate rows:  {len(test_candidates):,}")
print()
print(train_candidates["label"].value_counts())


Train candidate rows: 496,140
Test candidate rows:  240,605

label
0    396912
1     99228
Name: count, dtype: int64


## 10. Join candidate pairs with engineered features


In [15]:
def attach_features(
    candidates,
    user_features,
    streamer_features,
    pair_features,
):
    result = (
        candidates
        .merge(
            user_features,
            on="user_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            streamer_features,
            on="streamer",
            how="left",
            validate="many_to_one",
        )
        .merge(
            pair_features,
            on=["user_id", "streamer"],
            how="left",
            validate="many_to_one",
        )
    )

    pair_zero_cols = [
        "pair_interactions",
        "pair_watch_minutes",
        "pair_avg_watch_minutes",
        "pair_active_span_intervals",
        "pair_repeat_strength",
        "pair_interactions_last_6",
        "pair_interactions_last_18",
        "pair_interactions_last_42",
    ]

    for col in pair_zero_cols:
        result[col] = result[col].fillna(0)

    result["seen_before"] = (
        result["pair_interactions"] > 0
    ).astype("int8")

    max_recency = result["pair_recency_intervals"].max()
    recency_fill = (
        0
        if pd.isna(max_recency)
        else max_recency + 1
    )

    result["pair_recency_intervals"] = (
        result["pair_recency_intervals"]
        .fillna(recency_fill)
    )

    result["pair_interaction_share"] = (
        result["pair_interactions"]
        / result["user_interactions"].clip(lower=1)
    )

    result["pair_watch_share"] = (
        result["pair_watch_minutes"]
        / result["user_total_watch_minutes"].clip(lower=1)
    )

    return result


train_ranking = attach_features(
    train_candidates,
    train_user_features,
    train_streamer_features,
    train_pair_features,
)

test_ranking = attach_features(
    test_candidates,
    test_user_features,
    test_streamer_features,
    test_pair_features,
)

print("Train ranking shape:", train_ranking.shape)
print("Test ranking shape: ", test_ranking.shape)

train_ranking.head()


Train ranking shape: (496140, 43)
Test ranking shape:  (240605, 43)


,user_id,streamer,label,user_interactions,user_unique_streamers,user_total_watch_minutes,user_avg_watch_minutes,user_median_watch_minutes,user_first_start,user_last_start,user_active_span_intervals,user_last_activity_recency,user_repeat_rate,user_interactions_last_6_intervals,user_interactions_last_18_intervals,user_interactions_last_42_intervals,streamer_interactions,streamer_unique_viewers,streamer_total_watch_minutes,streamer_avg_watch_minutes,streamer_popularity_share,streamer_log_interactions,streamer_log_viewers,streamer_repeat_viewer_rate,streamer_interactions_last_6,streamer_interactions_last_18,streamer_interactions_last_42,streamer_interactions_prev_42,streamer_trend_ratio,pair_interactions,pair_watch_minutes,pair_avg_watch_minutes,pair_last_start,pair_first_start,pair_recency_intervals,pair_active_span_intervals,pair_repeat_strength,pair_interactions_last_6,pair_interactions_last_18,pair_interactions_last_42,seen_before,pair_interaction_share,pair_watch_share
0,1,esl_csgo,1,27,13,590,21.8519,10.0000,154,4327,4173,49,0.5185,0.0000,0.0000,0.0000,"11,454.0000","4,836.0000","366,670.0000",32.0124,0.0054,9.3462,8.4840,0.4957,8.0000,49.0000,172.0000,274.0000,0.6291,0.0000,0.0000,0.0000,NaN,NaN,"4,377.0000",0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,0.0000
1,1,h3x_tv,1,27,13,590,21.8519,10.0000,154,4327,4173,49,0.5185,0.0000,0.0000,0.0000,1.0000,1.0000,10.0000,10.0000,0.0000,0.6931,0.6931,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,NaN,NaN,"4,377.0000",0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,0.0000
2,1,jahrein,1,27,13,590,21.8519,10.0000,154,4327,4173,49,0.5185,0.0000,0.0000,0.0000,"1,679.0000",837.0000,"50,290.0000",29.9524,0.0008,7.4265,6.7310,0.4934,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,30.0000,30.0000,"3,757.0000","3,757.0000",619.0000,0.0000,0.6931,0.0000,0.0000,0.0000,1,0.0370,0.0508
3,1,jrokezftw,1,27,13,590,21.8519,10.0000,154,4327,4173,49,0.5185,0.0000,0.0000,0.0000,"1,032.0000",499.0000,"31,020.0000",30.0581,0.0005,6.9402,6.2146,0.4629,0.0000,0.0000,0.0000,30.0000,0.0323,1.0000,30.0000,30.0000,591.0000,591.0000,"3,785.0000",0.0000,0.6931,0.0000,0.0000,0.0000,1,0.0370,0.0508
4,1,ogrencievi,1,27,13,590,21.8519,10.0000,154,4327,4173,49,0.5185,0.0000,0.0000,0.0000,184.0000,119.0000,"4,250.0000",23.0978,0.0001,5.2204,4.7875,0.3697,0.0000,0.0000,0.0000,8.0000,0.1111,1.0000,10.0000,10.0000,"4,204.0000","4,204.0000",172.0000,0.0000,0.6931,0.0000,0.0000,0.0000,1,0.0370,0.0169


## 11. Add future diagnostics

These columns are useful for analysis only.

They are **future information** and must never be included as model features.


In [16]:
train_ranking = train_ranking.merge(
    train_positives[
        [
            "user_id",
            "streamer",
            "future_interactions",
            "future_watch_minutes",
        ]
    ],
    on=["user_id", "streamer"],
    how="left",
)

test_ranking = test_ranking.merge(
    test_positives[
        [
            "user_id",
            "streamer",
            "future_interactions",
            "future_watch_minutes",
        ]
    ],
    on=["user_id", "streamer"],
    how="left",
)

for frame in [train_ranking, test_ranking]:
    frame["future_interactions"] = (
        frame["future_interactions"].fillna(0)
    )
    frame["future_watch_minutes"] = (
        frame["future_watch_minutes"].fillna(0)
    )


## 12. Final feature set

We explicitly separate:

- identifiers
- model features
- target label
- future-only diagnostic columns


In [17]:
ID_COLUMNS = [
    "user_id",
    "streamer",
]

FEATURE_COLUMNS = [
    # User behavior
    "user_interactions",
    "user_unique_streamers",
    "user_total_watch_minutes",
    "user_avg_watch_minutes",
    "user_median_watch_minutes",
    "user_active_span_intervals",
    "user_last_activity_recency",
    "user_repeat_rate",
    "user_interactions_last_6_intervals",
    "user_interactions_last_18_intervals",
    "user_interactions_last_42_intervals",

    # Streamer behavior
    "streamer_interactions",
    "streamer_unique_viewers",
    "streamer_total_watch_minutes",
    "streamer_avg_watch_minutes",
    "streamer_popularity_share",
    "streamer_log_interactions",
    "streamer_log_viewers",
    "streamer_repeat_viewer_rate",
    "streamer_interactions_last_6",
    "streamer_interactions_last_18",
    "streamer_interactions_last_42",
    "streamer_interactions_prev_42",
    "streamer_trend_ratio",

    # User-streamer affinity
    "pair_interactions",
    "pair_watch_minutes",
    "pair_avg_watch_minutes",
    "pair_recency_intervals",
    "pair_active_span_intervals",
    "pair_repeat_strength",
    "pair_interactions_last_6",
    "pair_interactions_last_18",
    "pair_interactions_last_42",
    "seen_before",
    "pair_interaction_share",
    "pair_watch_share",
]

LABEL_COLUMN = "label"

FUTURE_ONLY_COLUMNS = [
    "future_interactions",
    "future_watch_minutes",
]

print(f"Number of model features: {len(FEATURE_COLUMNS)}")
FEATURE_COLUMNS


Number of model features: 36


['user_interactions',
 'user_unique_streamers',
 'user_total_watch_minutes',
 'user_avg_watch_minutes',
 'user_median_watch_minutes',
 'user_active_span_intervals',
 'user_last_activity_recency',
 'user_repeat_rate',
 'user_interactions_last_6_intervals',
 'user_interactions_last_18_intervals',
 'user_interactions_last_42_intervals',
 'streamer_interactions',
 'streamer_unique_viewers',
 'streamer_total_watch_minutes',
 'streamer_avg_watch_minutes',
 'streamer_popularity_share',
 'streamer_log_interactions',
 'streamer_log_viewers',
 'streamer_repeat_viewer_rate',
 'streamer_interactions_last_6',
 'streamer_interactions_last_18',
 'streamer_interactions_last_42',
 'streamer_interactions_prev_42',
 'streamer_trend_ratio',
 'pair_interactions',
 'pair_watch_minutes',
 'pair_avg_watch_minutes',
 'pair_recency_intervals',
 'pair_active_span_intervals',
 'pair_repeat_strength',
 'pair_interactions_last_6',
 'pair_interactions_last_18',
 'pair_interactions_last_42',
 'seen_before',
 'pair_in

## 13. Missing-value check

In [18]:
train_missing = (
    train_ranking[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

test_missing = (
    test_ranking[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Train missing values:")
display(train_missing[train_missing > 0])

print("Test missing values:")
display(test_missing[test_missing > 0])


Train missing values:


streamer_repeat_viewer_rate      6310
streamer_interactions            6310
streamer_interactions_prev_42    6310
streamer_interactions_last_42    6310
streamer_interactions_last_18    6310
streamer_interactions_last_6     6310
streamer_log_viewers             6310
streamer_log_interactions        6310
streamer_popularity_share        6310
streamer_avg_watch_minutes       6310
streamer_total_watch_minutes     6310
streamer_unique_viewers          6310
streamer_trend_ratio             6310
dtype: int64

Test missing values:


streamer_repeat_viewer_rate      2550
streamer_interactions            2550
streamer_interactions_prev_42    2550
streamer_interactions_last_42    2550
streamer_interactions_last_18    2550
streamer_interactions_last_6     2550
streamer_log_viewers             2550
streamer_log_interactions        2550
streamer_popularity_share        2550
streamer_avg_watch_minutes       2550
streamer_total_watch_minutes     2550
streamer_unique_viewers          2550
streamer_trend_ratio             2550
dtype: int64

## 14. Ranking-dataset sanity checks

In [19]:
train_positive_counts = (
    train_ranking.groupby("user_id")["label"].sum()
)

test_positive_counts = (
    test_ranking.groupby("user_id")["label"].sum()
)

assert (train_positive_counts >= 1).all()
assert (test_positive_counts >= 1).all()

assert not train_ranking.duplicated(
    ["user_id", "streamer"]
).any()

assert not test_ranking.duplicated(
    ["user_id", "streamer"]
).any()

assert set(
    train_ranking["label"].unique()
).issubset({0, 1})

assert set(
    test_ranking["label"].unique()
).issubset({0, 1})

print("All sanity checks passed.")


All sanity checks passed.


## 15. Inspect class balance and candidate-set size


In [20]:
def summarize_ranking_frame(frame, name):
    candidate_counts = frame.groupby("user_id").size()
    positive_counts = frame.groupby("user_id")["label"].sum()

    print(name)
    print("-" * len(name))
    print(f"Rows: {len(frame):,}")
    print(f"Users: {frame['user_id'].nunique():,}")
    print(f"Streamers: {frame['streamer'].nunique():,}")
    print(f"Positive rate: {frame['label'].mean():.2%}")
    print(
        f"Median candidates/user: "
        f"{candidate_counts.median():,.0f}"
    )
    print(
        f"Median positives/user: "
        f"{positive_counts.median():,.0f}"
    )
    print()


summarize_ranking_frame(
    train_ranking,
    "TRAIN RANKING DATA"
)

summarize_ranking_frame(
    test_ranking,
    "TEST RANKING DATA"
)


TRAIN RANKING DATA
------------------
Rows: 496,140
Users: 20,000
Streamers: 44,066
Positive rate: 20.00%
Median candidates/user: 15
Median positives/user: 3

TEST RANKING DATA
-----------------
Rows: 240,605
Users: 10,000
Streamers: 43,285
Positive rate: 20.00%
Median candidates/user: 15
Median positives/user: 3



## 16. Repeat-consumption diagnostic

Twitch is a repeat-consumption platform, so this is important.

We measure what percentage of future-positive user-streamer pairs were already seen in history.


In [21]:
def repeat_positive_rate(frame, name):
    positives = frame[
        frame["label"] == 1
    ]

    rate = positives["seen_before"].mean()

    print(
        f"{name}: {rate:.2%} of future-positive pairs "
        f"were already seen before."
    )


repeat_positive_rate(
    train_ranking,
    "Train"
)

repeat_positive_rate(
    test_ranking,
    "Test"
)


Train: 51.73% of future-positive pairs were already seen before.
Test: 54.87% of future-positive pairs were already seen before.


## 17. Inspect one user's candidate set


In [22]:
example_user = test_ranking["user_id"].iloc[0]

example_candidates = (
    test_ranking[
        test_ranking["user_id"] == example_user
    ]
    .sort_values(
        ["label", "streamer_interactions"],
        ascending=[False, False],
    )
)

print("Example user:", example_user)

display(
    example_candidates[
        [
            "user_id",
            "streamer",
            "label",
            "seen_before",
            "pair_interactions",
            "pair_watch_minutes",
            "pair_recency_intervals",
            "pair_interaction_share",
            "streamer_interactions",
            "streamer_trend_ratio",
        ]
    ].head(30)
)


Example user: 1


,user_id,streamer,label,seen_before,pair_interactions,pair_watch_minutes,pair_recency_intervals,pair_interaction_share,streamer_interactions,streamer_trend_ratio
4,1,mithrain,1,1,7.0000,90.0000,973.0000,0.1795,"4,950.0000",1.0000
6,1,wtcn,1,1,4.0000,70.0000,421.0000,0.1026,"3,301.0000",1.0000
3,1,kendinemuzisyen,1,1,3.0000,180.0000,"1,688.0000",0.0769,"3,247.0000",1.0000
1,1,jahrein,1,1,2.0000,50.0000,423.0000,0.0513,"2,286.0000",1.0000
0,1,grimnax,1,0,0.0000,0.0000,"5,289.0000",0.0000,777.0000,21.0000
2,1,jtgtv,1,0,0.0000,0.0000,"5,289.0000",0.0000,217.0000,77.0000
5,1,towshun,1,1,1.0000,40.0000,93.0000,0.0256,27.0000,0.5000
48130,1,innocents,0,0,0.0000,0.0000,"5,289.0000",0.0000,996.0000,0.1000
48133,1,wtf_winds123,0,0,0.0000,0.0000,"5,289.0000",0.0000,746.0000,0.0500
48148,1,floyl,0,0,0.0000,0.0000,"5,289.0000",0.0000,74.0000,1.0000


## 18. Save processed ranking tables

These files are the inputs to Notebook 03.


In [23]:
OUTPUT_DIR = Path("../data/processed/twitch")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

train_output = (
    OUTPUT_DIR
    / "twitch_train_ranking.csv.gz"
)

test_output = (
    OUTPUT_DIR
    / "twitch_test_ranking.csv.gz"
)

columns_to_save = (
    ID_COLUMNS
    + FEATURE_COLUMNS
    + [LABEL_COLUMN]
    + FUTURE_ONLY_COLUMNS
)

train_ranking[
    columns_to_save
].to_csv(
    train_output,
    index=False,
    compression="gzip",
)

test_ranking[
    columns_to_save
].to_csv(
    test_output,
    index=False,
    compression="gzip",
)

print("Saved:")
print(train_output)
print(test_output)


Saved:
../data/processed/twitch/twitch_train_ranking.csv.gz
../data/processed/twitch/twitch_test_ranking.csv.gz


In [24]:
# ============================================================
# SAVE NOTEBOOK 02 RESULTS SAFELY
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path(
    "../results/02_twitch_ranking_data_preparation"
)

TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Save important result tables only if they exist
# ------------------------------------------------------------

possible_tables = [
    "split_summary",
    "coverage_summary",
    "feature_table_summary",
    "positive_summary",
    "feature_list",
    "missing_summary",
    "ranking_summary",
    "repeat_positive_summary",
    "example_candidates",
]

saved_tables = []

for name in possible_tables:

    if name in globals():

        obj = globals()[name]

        if isinstance(
            obj,
            (pd.DataFrame, pd.Series)
        ):

            output_path = (
                TABLES_DIR /
                f"{name}.csv"
            )

            if isinstance(
                obj,
                pd.DataFrame
            ):
                obj.to_csv(
                    output_path,
                    index=False
                )

            else:
                obj.to_csv(
                    output_path,
                    header=True
                )

            saved_tables.append(
                output_path.name
            )


# ------------------------------------------------------------
# Save currently open figures
# ------------------------------------------------------------

saved_figures = []

for fig_num in plt.get_fignums():

    fig = plt.figure(fig_num)

    output_path = (
        FIGURES_DIR /
        f"figure_{fig_num:02d}.png"
    )

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    saved_figures.append(
        output_path.name
    )


# ------------------------------------------------------------
# Build key metrics directly from existing ranking tables
# ------------------------------------------------------------

metrics = []

if "train_ranking" in globals():

    metrics.extend([
        {
            "metric": "train_rows",
            "value": len(train_ranking)
        },
        {
            "metric": "train_users",
            "value": train_ranking["user_id"].nunique()
        },
        {
            "metric": "train_positive_rate",
            "value": train_ranking["label"].mean()
        },
    ])

    if "seen_before" in train_ranking.columns:

        metrics.append(
            {
                "metric":
                    "train_repeat_positive_rate",

                "value":
                    train_ranking.loc[
                        train_ranking["label"] == 1,
                        "seen_before"
                    ].mean()
            }
        )


if "test_ranking" in globals():

    metrics.extend([
        {
            "metric": "test_rows",
            "value": len(test_ranking)
        },
        {
            "metric": "test_users",
            "value": test_ranking["user_id"].nunique()
        },
        {
            "metric": "test_positive_rate",
            "value": test_ranking["label"].mean()
        },
    ])

    if "seen_before" in test_ranking.columns:

        metrics.append(
            {
                "metric":
                    "test_repeat_positive_rate",

                "value":
                    test_ranking.loc[
                        test_ranking["label"] == 1,
                        "seen_before"
                    ].mean()
            }
        )


if "FEATURE_COLUMNS" in globals():

    metrics.append(
        {
            "metric": "feature_count",
            "value": len(FEATURE_COLUMNS)
        }
    )


key_metrics = pd.DataFrame(metrics)

key_metrics.to_csv(
    RESULTS_DIR / "key_metrics.csv",
    index=False
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print(
    "Results saved to:",
    RESULTS_DIR.resolve()
)

print(
    f"Tables saved: {len(saved_tables)}"
)

print(
    f"Figures saved: {len(saved_figures)}"
)

print(
    "Key metrics saved:",
    len(key_metrics)
)

Results saved to: /Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/results/02_twitch_ranking_data_preparation
Tables saved: 1
Figures saved: 0
Key metrics saved: 9


## 19. Final outcome

We have now converted raw Twitch viewing logs into a ranking-ready supervised dataset.

```text
historical Twitch interactions
        ↓
chronological cutoffs
        ↓
user behavior features
streamer popularity/trend features
user-streamer affinity features
recency/repeat-consumption features
        ↓
future positive streamers
+
sampled negative candidates
        ↓
(user, candidate, features, label)
```

### Next notebook

**03_twitch_baseline_ranker.ipynb**

That notebook will:

1. build a popularity baseline
2. train a simple probabilistic ranking model
3. generate inference scores
4. rank candidates for each user
5. calculate Recall@K, MRR, and NDCG@K
6. compare single-candidate vs batched inference
